# 03 — Streaming Transformations (Bronze → Silver)

Reads from the Bronze table as a stream, applies business logic
transformations, and writes to a Silver Delta table.

In [1]:
from databricks.connect import DatabricksSession
from pyspark.sql import functions as F
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "streaming_demo")

BRONZE_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.slot_events_bronze"
SILVER_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.slot_events_silver"
CHECKPOINT_PATH = f"/Volumes/{UC_CATALOG}/{UC_SCHEMA}/casino_raw_events/_checkpoints/silver"

## Read Bronze as a stream and apply transformations

In [2]:
bronze_stream = spark.readStream.table(BRONZE_TABLE)

silver_stream = (
    bronze_stream
    # Cast string columns to proper types (Auto Loader infers JSON values as strings)
    .withColumn("bet_amount", F.col("bet_amount").cast("double"))
    .withColumn("win_amount", F.col("win_amount").cast("double"))
    # Parse the event timestamp
    .withColumn("event_ts", F.to_timestamp("event_timestamp"))
    # Calculate net outcome (positive = player won, negative = house won)
    .withColumn("net_outcome", F.round(F.col("win_amount") - F.col("bet_amount"), 2))
    # Categorize the bet size for easy segmentation
    .withColumn(
        "bet_tier",
        F.when(F.col("bet_amount") < 10, "Low")
        .when(F.col("bet_amount") < 100, "Medium")
        .when(F.col("bet_amount") < 500, "High")
        .otherwise("Whale")
    )
    # Flag whether the play was a win or loss
    .withColumn("is_win", F.col("win_amount") > 0)
    # Extract hour for time-of-day analysis
    .withColumn("event_hour", F.hour("event_ts"))
    # Drop the original string timestamp and rescued data column
    .drop("event_timestamp", "_rescued_data")
)

print("Silver schema:")
silver_stream.printSchema()

Silver schema:
root
 |-- bet_amount: double (nullable = true)
 |-- casino_floor: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- game_type: string (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- player_id: string (nullable = true)
 |-- win_amount: double (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- net_outcome: double (nullable = true)
 |-- bet_tier: string (nullable = false)
 |-- is_win: boolean (nullable = true)
 |-- event_hour: integer (nullable = true)



## Write to Silver Delta table

In [3]:
query = (
    silver_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .toTable(SILVER_TABLE)
)

query.awaitTermination()
print(f"Silver table ready: {SILVER_TABLE}")

Silver table ready: alexander_booth.streaming_demo.slot_events_silver


## Preview Silver data

In [4]:
silver_df = spark.table(SILVER_TABLE)
print(f"Silver table row count: {silver_df.count()}")
silver_df.show(10, truncate=False)

Silver table row count: 500
+----------+------------+------------------------------------+--------------------+----------+---------+----------+-------------------+-----------+--------+------+----------+
|bet_amount|casino_floor|event_id                            |game_type           |machine_id|player_id|win_amount|event_ts           |net_outcome|bet_tier|is_win|event_hour|
+----------+------------+------------------------------------+--------------------+----------+---------+----------+-------------------+-----------+--------+------+----------+
|200.34    |High Roller |48582eb2-71a0-41e5-813a-caf352ddda77|Electronic Blackjack|MCH-0020  |PLR-23396|0.0       |2026-03-25 10:06:35|-200.34    |High    |false |10        |
|353.93    |Sports Zone |f17eb186-bb0f-45c5-93f2-48514d5804c5|Electronic Roulette |MCH-0023  |PLR-38785|436.57    |2026-03-25 10:06:30|82.64      |High    |true  |10        |
|147.46    |VIP Lounge  |b5a5e8b1-ba35-4435-a1e3-60752f7ae2fe|Electronic Roulette |MCH-0026  |PLR